# Análisis de sensibilidad de variantes NO-PASS en genes candidatos MVA

**MVA Hackathon 2026 · Sensibilidad del filtrado VCF**

Este notebook evalúa las variantes clasificadas como `NO-PASS` por el filtrado técnico del VCF en los 6 genes candidatos de MVA

El objetivo es determinar si el filtrado estricto de variantes `PASS` podría haber excluido variantes potencialmente relevantes para el caso clínico.

El análisis utiliza la misma lista de genes candidatos y las mismas coordenadas utilizadas en `02_filtrado_vcf_genes_mva.ipynb`, pero conserva únicamente las variantes cuyo campo `FILTER` es diferente de `PASS`.

In [ ]:
import gzip
import json
import requests
import pandas as pd
from pathlib import Path

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw")
RESULTS_DIR = Path("../results")
VCF = DATA_DIR / "WGS_EX2312012_HGWCNDSX7.vcf.gz"

RESULTS_DIR.mkdir(exist_ok=True)

print("VCF:", VCF)
print("Resultados:", RESULTS_DIR)

In [ ]:
FALLBACK_COORDS = {
    "BUB1B":  ("15", 40_160_984,  40_221_137),
    "CEP57":  ("11", 95_789_965,  95_837_070),
    "TRIP13": ("5",    892_849,      919_357),
    "BUB1":   ("2",  110_143_783, 110_244_769),
    "MAD1L1": ("7",   1_815_787,   2_233_243),
    "MAD2L1": ("4",  120_055_623, 120_067_192),
}

PADDING = 5_000

def fetch_ensembl(gene):
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?content-type=application/json"
    try:
        d = requests.get(url, timeout=10).json()
        return str(d["seq_region_name"]), int(d["start"]), int(d["end"])
    except Exception:
        return None

gene_coords = {}

for gene, fb in FALLBACK_COORDS.items():
    result = fetch_ensembl(gene)

    if result:
        gene_coords[gene] = result
        print(
            f"{gene:8s}  {result[0]}:{result[1]:>12,}-{result[2]:>12,}  (Ensembl)"
        )
    else:
        gene_coords[gene] = fb
        print(
            f"{gene:8s}  {fb[0]}:{fb[1]:>12,}-{fb[2]:>12,}  (fallback)"
        )

with open(RESULTS_DIR / "mva_gene_coords_grch38.json", "w") as f:
    json.dump(gene_coords, f, indent=2)

In [ ]:
# Construir índice cromosoma → lista de intervalos de genes candidatos

interval_idx = {}

for gene, (chrom, start, end) in gene_coords.items():
    interval_idx.setdefault(chrom, []).append(
        (start - PADDING, end + PADDING, gene)
    )


def gene_at(chrom, pos):
    for start, end, gene in interval_idx.get(chrom, []):
        if start <= pos <= end:
            return gene
    return None

In [ ]:
def classify_zygosity(gt):
    alleles = gt.replace("|", "/").split("/")

    if "." in alleles:
        return "MISSING"

    if set(alleles) == {"0"}:
        return "HOM_REF"

    if alleles[0] == alleles[1] and alleles[0] != "0":
        return "HOM_ALT"

    if "0" in alleles:
        return "HET"

    return "OTHER"

In [ ]:
hits = []

print(f"Escaneando {VCF.name} ...")

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info, fmt = p[:9]
        sample = p[9] if len(p) > 9 else ""

        # Sensibilidad: conservar únicamente variantes NO-PASS
        if filt == "PASS":
            continue

        pos = int(pos_s)
        gene = gene_at(chrom, pos)

        if not gene:
            continue

        gt_dict = dict(zip(fmt.split(":"), sample.split(":")))

        gt = gt_dict.get("GT", ".")
        dp = int(gt_dict.get("DP", 0) or 0)
        gq = int(gt_dict.get("GQ", 0) or 0)
        ad = gt_dict.get("AD", ".")

        var_type = (
            "SNV" if len(ref) == 1 and len(alt) == 1
            else "DEL" if len(ref) > len(alt)
            else "INS"
        )

        hits.append({
            "gene": gene,
            "chrom": chrom,
            "pos": pos,
            "id": vid,
            "ref": ref,
            "alt": alt,
            "qual": float(qual),
            "filter": filt,
            "type": var_type,
            "zygosity": classify_zygosity(gt),
            "GT": gt,
            "DP": dp,
            "GQ": gq,
            "AD": ad,
        })

df_nonpass = pd.DataFrame(hits)

df_nonpass.to_csv(
    RESULTS_DIR / "variants_mva_genes_nonpass.tsv",
    sep="\t",
    index=False
)

print(f"Total variantes NO-PASS en genes candidatos: {len(df_nonpass):,}")

In [ ]:
print("=== TODAS LAS VARIANTES NO-PASS EN GENES CANDIDATOS ===")
print(df_nonpass.to_string(index=False))

print("\n=== Variantes NO-PASS por gen ===")
print(
    df_nonpass.groupby("gene")[["pos"]]
    .count()
    .rename(columns={"pos": "total"})
    .sort_values("total", ascending=False)
)

print("\n=== Por gen y zigosidad ===")
print(pd.crosstab(df_nonpass["gene"], df_nonpass["zygosity"]))

print("\n=== Por tipo de variante ===")
print(
    df_nonpass.groupby(["gene", "type"])["pos"]
    .count()
    .unstack(fill_value=0)
)

print("\n=== Motivo(s) de NO-PASS ===")
print(df_nonpass["filter"].value_counts())

## Interpretación del análisis de sensibilidad

El filtrado principal del VCF conserva únicamente variantes `PASS`. Este análisis de sensibilidad evalúa las variantes `NO-PASS` localizadas en los genes candidatos de MVA para determinar si alguna variante potencialmente relevante podría haber sido excluida por los filtros técnicos.

Las variantes `NO-PASS` no se consideran patogénicas por este motivo. El objetivo es caracterizarlas según:

- gen afectado;
- tipo de variante;
- cigosidad;
- calidad de la variante (`QUAL`);
- profundidad de secuenciación (`DP`);
- calidad del genotipo (`GQ`);
- balance de alelos (`AD`);
- motivo por el cual la variante no fue clasificada como `PASS`.

Este análisis permite identificar variantes que podrían requerir una revisión adicional, especialmente en genes candidatos relevantes para el fenotipo.

In [ ]:
# Mostrar los campos INFO de las variantes NO-PASS

print("=== INFO de las variantes NO-PASS ===")

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info = p[:8]

        if filt == "PASS":
            continue

        pos = int(pos_s)
        gene = gene_at(chrom, pos)

        if gene:
            print(
                f"{gene:7s} {chrom}:{pos} | "
                f"FILTER={filt} | "
                f"QUAL={qual} | "
                f"INFO={info}"
            )

In [ ]:
# Resumen técnico de las variantes NO-PASS

summary_nonpass = df_nonpass[
    ["gene", "chrom", "pos", "ref", "alt",
     "qual", "filter", "zygosity", "GT", "DP", "GQ", "AD"]
].copy()

summary_nonpass

In [ ]:
# Extraer métricas técnicas del campo INFO

def parse_info(info):
    values = {}

    for item in info.split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            values[key] = value

    return values


technical_hits = []

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info = p[:8]

        if filt == "PASS":
            continue

        pos = int(pos_s)
        gene = gene_at(chrom, pos)

        if not gene:
            continue

        info_dict = parse_info(info)

        technical_hits.append({
            "gene": gene,
            "chrom": chrom,
            "pos": pos,
            "id": vid,
            "ref": ref,
            "alt": alt,
            "QUAL": float(qual),
            "FILTER": filt,
            "DP": float(info_dict.get("DP", "nan")),
            "QD": float(info_dict.get("QD", "nan")),
            "MQ": float(info_dict.get("MQ", "nan")),
            "FS": float(info_dict.get("FS", "nan")),
            "MQRankSum": float(info_dict.get("MQRankSum", "nan")),
            "ReadPosRankSum": float(info_dict.get("ReadPosRankSum", "nan")),
            "SOR": float(info_dict.get("SOR", "nan")),
        })

df_technical = pd.DataFrame(technical_hits)

df_technical.reset_index(drop=True)